In [1]:
print("all okay")

all okay


In [2]:
from __future__ import annotations
from pathlib import Path
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader

C:\Users\samba\AppData\Local\Temp\ipykernel_15340\583762284.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
d:\New_file\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

In [4]:
COLLECTION_NAME = "company-policy-rag"

In [21]:
# --------------------------------------------------
# 3. Load PDF
# --------------------------------------------------
file_path = r"D:\New_file\llama2-research-paper.pdf"
pdf_loader = PyPDFLoader(file_path)
pdf_data = pdf_loader.load()
print("Total pdf_data:", len(pdf_data))

Total pdf_data: 77


In [22]:
# --------------------------------------------------
# 4. Create chunks
# --------------------------------------------------
chunker = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)
chunked_data = chunker.split_documents(pdf_data)
print("Total chunked_data:", len(chunked_data))

Total chunked_data: 175


In [23]:
# 3. Add useful metadata
for chunk_index, chunk in enumerate(chunked_data):
    chunk.metadata["chunk_id"] = chunk_index
    chunk.metadata["filename"] = "llama2-research-paper.pdf"

In [24]:
from langchain_huggingface import HuggingFaceEmbeddings

In [25]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [16]:
dimension = len(
    embeddings.embed_query("dimension check")
)
print("Embedding dimension:", dimension)

dimension = len(
    embeddings.embed_query("dimension check")
)

Embedding dimension: 384


In [ ]:
# # 5. Local Qdrant
# client = QdrantClient(
#     path=str(Path(__file__).parent / "qdrant_data")
# )

In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
qdrant_api_key = os.getenv("Qdrant_API_KEY")
qdrant_cluster_endpoint = os.getenv("Qdrant_cluster_endpoint")

In [17]:
client = QdrantClient(api_key=qdrant_api_key,url=qdrant_cluster_endpoint)

In [18]:
# 6. Create collection
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=dimension,
            distance=models.Distance.COSINE,
        ),
    )

In [19]:
# 7. LangChain Qdrant vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

In [29]:
# 8. Use deterministic or stable IDs in production
chunk_ids = [
    str(uuid4())
    for _ in chunked_data
]

In [30]:
# 9. Add documents
inserted_ids = vector_store.add_documents(
    documents=chunked_data,
    ids=chunk_ids,
)

print("Inserted chunked_data:", len(inserted_ids))

Inserted chunked_data: 175


In [31]:
query = "What is the annual leave policy?"

results = vector_store.similarity_search(
    query=query,
    k=4,
)

for rank, document in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Content:", document.page_content)
    print("Metadata:", document.metadata)


Result 1
Content: and 100 outputs isT ∈ [1.2, 1.3]. Given a finite compute budget, it is therefore necessary to re-adjust the
temperature progressively. Note that this temperature rescaling happens for a constant number of steps for
each model, and always starting from the base model on each new RLHF version.
PPO. WefurthertrainourlanguagemodelfollowingtheRLschemeofStiennonetal.(2020),whichusesthe
reward model as an estimate for the true reward function (human preference) and the pretrained language
model as the policy to optimize. During this phase, we seek to optimize the following objective:
arg max
π
Ep∼D,g∼π[R(g | p)] (3)
We iteratively improve the policy by sampling promptsp from our datasetD and generationsg from the
policy π and use the PPO algorithm and loss function to achieve this objective.
The final reward function we use during optimization,
R(g | p) = ˜Rc(g | p) − βDKL (πθ(g | p) ∥ π0(g | p)) (4)
contains a penalty term for diverging from the original policyπ0. As was o